# Q1D | Train Linear Regression | Individual Stage Experiments (impute, transform, scale)

# Imports

In [1]:
import os
import re
import warnings

import sklearn
print(sklearn.__version__)

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor

from Q1.consts import *
from utils.read_csv_to_df import read_csv_to_df
from Q1.utils.create_pipeline import create_preprocessing, display_pipeline_grid
from Q1.utils.run_experiment import build_pipelines, run_experiment
from Q1.utils.run_combinations import skew_aware_combinations, combination_code, combination_pipeline, run_combinations

from Q1.models import MODELS
from Q1.performance_measures import SCORERS
from Q1.utils.preprocessing.scale import StandardScaler, MinMaxScaler
from Q1.utils.preprocessing.impute import SimpleImputer, KNNImputer, IterativeImputer
from Q1.utils.preprocessing.transform import PowerTransformer, FunctionTransformer, MinShift, raise_to_power, on_columns

1.3.2


# Defaults

In [2]:
SEED = 42
np.random.seed(SEED)
pd.set_option("display.float_format", "{:,.4f}".format)

# shared by every experiment so each one is scored on the exact same folds
CV = KFold(n_splits=4, shuffle=True, random_state=SEED)

# IterativeImputer with a KNN or tree estimator never meets its stopping rule (its fills jump between rounds
# instead of settling), so this warning is expected for those variants and is hidden.
# filterwarnings covers this process (the final refit); PYTHONWARNINGS covers the n_jobs=-1 worker processes,
# which only read it when they start, so restart the kernel if this cell is changed after an experiment has run
IMPUTER_WARNING = "[IterativeImputer] Early stopping criterion not reached"
warnings.filterwarnings("ignore", message=re.escape(IMPUTER_WARNING))
os.environ["PYTHONWARNINGS"] = f"ignore:{IMPUTER_WARNING}"

# Fetch Data

> Only the training set is used here. The test set is held back so it plays no part in choosing the model.

In [3]:
train_df = read_csv_to_df(path=TRAIN_PATH)

X = train_df.drop(columns=TARGET)
y = train_df[TARGET]

# Experiment A | Base

> No PreProcessing

In [4]:
experimentA_model_output_dir = MODEL_OUTPUT_DIR / 'A-BASE-DropRowsWithMissingValues'
experimentA_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> since linear regression can not accept missing values we simply drop all rows with missing values in this base experiment

In [5]:
def drop_missing_rows(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    mask = X.notna().all(axis=1)
    return X[mask], y[mask]

X_experimentA_clean, y_experimentA_clean = drop_missing_rows(X, y)

## PreProcessing

| impute      | transform   | scale       |
|-------------|-------------|-------------|
| passthrough | passthrough | passthrough |

In [6]:
experimentA_preProcessing = [
	create_preprocessing(preprocessing={})
]

## Build Pipelines

In [7]:
experimentA_pipelines = build_pipelines(preprocessings=experimentA_preProcessing, models=MODELS, seed=SEED)
# display_pipeline_grid(experimentA_pipelines)

## Cross-Validation Model Fitting

> cross_validate is a function that fits immediately; GridSearchCV is an object that waits for .fit()

In [8]:
experimentA_df = run_experiment(
    experimentA_pipelines,
    X=X_experimentA_clean,
    y=y_experimentA_clean,
    cv=CV,
    scorers=SCORERS,
    output_dir=experimentA_model_output_dir,
    experiment="A",
    cleaning="drop_missing_rows",
)
experimentA_df

/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the numb

,experiment,model,cleaning,impute,transform,scale,rmse,mse,mae,path
0,A,ridge,drop_missing_rows,none,none,none,0.5460,0.2990,0.4230,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
1,A,linear_regression,drop_missing_rows,none,none,none,0.5470,0.2990,0.4220,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
2,A,svr_linear,drop_missing_rows,none,none,none,0.5500,0.3030,0.4170,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
3,A,linear_svr,drop_missing_rows,none,none,none,0.6140,0.3810,0.4730,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
4,A,elastic_net,drop_missing_rows,none,none,none,0.7370,0.5440,0.6030,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
5,A,lasso,drop_missing_rows,none,none,none,0.7450,0.5560,0.6090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
6,A,sgd,drop_missing_rows,none,none,none,"194,415,414,023.5250","56,597,954,438,661,410,914,304.0000","192,150,942,073.5320",/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Experiment A Results

- Ridge (RMSE 0.546, MAE 0.423) and linear_regression (RMSE 0.547, MAE 0.422) set the baseline. svr_linear (0.550) is within the 0.002 noise floor of them.
- SGD diverged (running on unscaled features): instead of settling on good weights, its updates grew larger each step until the predictions were astronomically wrong (RMSE about 1.9e11). SGD takes gradient steps proportional to feature values, so Healthy life expectancy (32 to 77), sitting next to features that are mostly under 1, produces huge steps that overshoot and blow up.
- LinearSVR did not converge (5 Liblinear ConvergenceWarnings: all 4 folds plus the final fit) and scored 0.614, about 0.07 worse than OLS.
- Lasso (0.745) and ElasticNet (0.737) at the default alpha=1 underfit by about 0.2.
- **Carry forward**
  - 0.547 (linear_regression) / 0.546 (ridge) is the bar: a combined pipeline has to beat it by more than 0.002 to count as an improvement.
  - SGD and LinearSVR scores are meaningless without scaling, so every combined pipeline that uses them needs a scale step.

# Experiment B | Scaling Only

In [9]:
experimentB_model_output_dir = MODEL_OUTPUT_DIR / 'B-SCALING-DropRowsWithMissingValues'
experimentB_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> since linear regression can not accept missing values we simply drop all rows with missing values in this base experiment

In [10]:
X_experimentB_clean, y_experimentB_clean = drop_missing_rows(X, y)

## PreProcessing

| impute      | transform   | scale            |
|-------------|-------------|------------------|
| passthrough | passthrough | standard, minmax |

In [11]:
experimentB_preProcessing = [
	create_preprocessing(preprocessing={ "scale": StandardScaler() }),
	create_preprocessing(preprocessing={ "scale": MinMaxScaler() }),
]

## Build Pipelines

In [12]:
experimentB_pipelines = build_pipelines(preprocessings=experimentB_preProcessing, models=MODELS, seed=SEED)
# display_pipeline_grid(experimentB_pipelines)

## Cross-Validation Model Fitting

> cross_validate is a function that fits immediately; GridSearchCV is an object that waits for .fit()

In [13]:
experimentB_df = run_experiment(
    experimentB_pipelines,
    X=X_experimentB_clean,
    y=y_experimentB_clean,
    cv=CV,
    scorers=SCORERS,
    output_dir=experimentB_model_output_dir,
    experiment="B",
    cleaning="drop_missing_rows",
)
experimentB_df

,experiment,model,cleaning,impute,transform,scale,rmse,mse,mae,path
0,B,ridge,drop_missing_rows,none,none,MinMaxScaler,0.5460,0.2990,0.4220,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
1,B,ridge,drop_missing_rows,none,none,StandardScaler,0.5470,0.2990,0.4220,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
2,B,linear_regression,drop_missing_rows,none,none,StandardScaler,0.5470,0.2990,0.4220,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
3,B,linear_regression,drop_missing_rows,none,none,MinMaxScaler,0.5470,0.2990,0.4220,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
4,B,sgd,drop_missing_rows,none,none,StandardScaler,0.5470,0.2990,0.4220,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
5,B,svr_linear,drop_missing_rows,none,none,MinMaxScaler,0.5510,0.3040,0.4180,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
6,B,svr_linear,drop_missing_rows,none,none,StandardScaler,0.5520,0.3050,0.4180,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
7,B,linear_svr,drop_missing_rows,none,none,StandardScaler,0.5530,0.3060,0.4190,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
8,B,linear_svr,drop_missing_rows,none,none,MinMaxScaler,0.5530,0.3060,0.4190,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
9,B,sgd,drop_missing_rows,none,none,MinMaxScaler,0.5610,0.3150,0.4330,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Experiment B Results

- linear_regression is unchanged by either scaler (RMSE 0.547, MAE 0.422, same as A): OLS predictions don't depend on feature scale. Ridge only moves within rounding (0.546 MinMaxScaler, 0.547 StandardScaler): its penalty does depend on scale, but alpha=1 is tiny next to 1,374 rows, so the shift stays inside the noise floor.
- Scaling fixes the models that broke in A: SGD goes from diverged to 0.547 (StandardScaler) and 0.561 (MinMaxScaler), and LinearSVR converges (no warnings) and improves from 0.614 to 0.553 with either scaler. svr_linear stays at 0.551 / 0.552.
- Lasso collapses to predicting the mean with both scalers: its RMSE of 1.128 equals the target's standard deviation on these rows (1.128), so every coefficient was shrunk to 0. ElasticNet collapses the same way with MinMaxScaler (1.128) and nearly does with StandardScaler (0.881). On unscaled data (A) the large raw scale of Healthy life expectancy kept its coefficient small enough to survive the penalty; once every feature is on a unit scale, alpha=1 wipes them all out.
- Neither scaler wins: the two differ by 0.001 or less for every model except SGD (0.014 in favor of StandardScaler) and ElasticNet.
- **Carry forward**
  - Every combined pipeline needs a scale step for SGD, LinearSVR, Lasso, and ElasticNet. StandardScaler is the safer default for SGD.
  - Lasso and ElasticNet need alpha tuned (values well below 1), or they will rank last for the wrong reason.
  - Scaling alone does not improve the best RMSE (0.546 / 0.547), so any gain has to come from imputing or transforming.

# Experiment C | Impute Only

In [14]:
experimentC_model_output_dir = MODEL_OUTPUT_DIR / 'C-IMPUTE-KeepAllRows'
experimentC_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> no rows are dropped in this experiment. The imputer fills the missing values instead, and because it is a pipeline step it is refit on the training folds only, so the validation fold never leaks into the fill values

In [15]:
X_experimentC_clean, y_experimentC_clean = X, y

## PreProcessing

| impute                                                                                   | transform   | scale       |
|------------------------------------------------------------------------------------------|-------------|-------------|
| simple (mean, median), knn (3 neighbor counts × 2 weightings), iterative (3 estimators)  | passthrough | passthrough |

> - **SimpleImputer:** mean and median. Median suits the 6 of 8 features with |skew| ≥ 0.5 (Q1B); mean is the default for comparison
> - **KNNImputer:** `n_neighbors` 3, 5, 10 × `weights` uniform (every neighbor counts equally) and distance (closer neighbors count more)
> - **IterativeImputer:** `initial_strategy` median, with `estimator` BayesianRidge (the default, linear), KNeighborsRegressor, and ExtraTreesRegressor (both can follow non-linear shapes like the corruption L shape)

In [16]:
experimentC_preProcessing = [
	create_preprocessing(preprocessing={ "impute": SimpleImputer(strategy="mean") }),
	create_preprocessing(preprocessing={ "impute": SimpleImputer(strategy="median") }),
	# create_preprocessing(preprocessing={ "impute": SimpleImputer(strategy="most_frequent") }), # not a good fit for continuous values

	create_preprocessing(preprocessing={ "impute": KNNImputer(n_neighbors=3, weights="uniform") }),
	create_preprocessing(preprocessing={ "impute": KNNImputer(n_neighbors=3, weights="distance") }),
	create_preprocessing(preprocessing={ "impute": KNNImputer(n_neighbors=5, weights="uniform") }),
	create_preprocessing(preprocessing={ "impute": KNNImputer(n_neighbors=5, weights="distance") }),
	create_preprocessing(preprocessing={ "impute": KNNImputer(n_neighbors=10, weights="uniform") }),
	create_preprocessing(preprocessing={ "impute": KNNImputer(n_neighbors=10, weights="distance") }),

	# BayesianRidge is the default estimator, passed explicitly so the label names it instead of showing None
	create_preprocessing(preprocessing={ "impute": IterativeImputer(estimator=BayesianRidge(), initial_strategy="median", random_state=SEED) }),
	create_preprocessing(preprocessing={ "impute": IterativeImputer(estimator=KNeighborsRegressor(), initial_strategy="median", random_state=SEED) }),
	create_preprocessing(preprocessing={ "impute": IterativeImputer(estimator=ExtraTreesRegressor(random_state=SEED), initial_strategy="median", random_state=SEED) }),
]

## Build Pipelines

In [17]:
experimentC_pipelines = build_pipelines(preprocessings=experimentC_preProcessing, models=MODELS, seed=SEED)
# display_pipeline_grid(experimentC_pipelines)

## Cross-Validation Model Fitting

> cross_validate is a function that fits immediately; GridSearchCV is an object that waits for .fit()

In [18]:
experimentC_df = run_experiment(
    experimentC_pipelines,
    X=X_experimentC_clean,
    y=y_experimentC_clean,
    cv=CV,
    scorers=SCORERS,
    output_dir=experimentC_model_output_dir,
    experiment="C",
    cleaning="none",
)
experimentC_df

/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the numb

,experiment,model,cleaning,impute,transform,scale,rmse,mse,mae,path
0,C,linear_regression,none,"KNNImputer(n_neighbors=3,weights=uniform)",none,none,0.5470,0.3000,0.4240,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
1,C,linear_regression,none,"KNNImputer(n_neighbors=10,weights=distance)",none,none,0.5480,0.3010,0.4250,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
2,C,linear_regression,none,"KNNImputer(n_neighbors=10,weights=uniform)",none,none,0.5480,0.3010,0.4250,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
3,C,linear_regression,none,"KNNImputer(n_neighbors=3,weights=distance)",none,none,0.5480,0.3010,0.4250,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
4,C,linear_regression,none,"KNNImputer(n_neighbors=5,weights=uniform)",none,none,0.5480,0.3010,0.4250,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
...,...,...,...,...,...,...,...,...,...,...
72,C,sgd,none,"KNNImputer(n_neighbors=3,weights=distance)",none,none,"58,800,973,178.7000","4,804,140,803,520,112,623,616.0000","57,669,395,367.6710",/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
73,C,sgd,none,IterativeImputer(estimator=ExtraTreesRegressor...,none,none,"156,854,451,125.2990","83,170,541,223,290,286,899,200.0000","155,080,653,866.3240",/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
74,C,sgd,none,"KNNImputer(n_neighbors=5,weights=uniform)",none,none,"243,237,587,953.2760","127,851,930,715,248,387,948,544.0000","241,148,233,871.9700",/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
75,C,sgd,none,IterativeImputer(estimator=KNeighborsRegressor...,none,none,"259,434,354,497.2450","252,843,345,831,445,852,585,984.0000","256,992,653,709.8020",/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Experiment C Results

- C keeps all 1,559 rows (185 more than A), so its validation folds include imputed rows. Its scores are not a like-for-like comparison with A, which is scored on the 1,374 complete rows only.
- The output table shows only its first 5 and last 5 of 77 rows. The best 5 are all linear_regression with KNNImputer: 0.547 (3 neighbors, uniform) and 0.548 (10 distance, 10 uniform, 3 distance, 5 uniform), MAE 0.424 to 0.425. They tie each other and A (0.547) within the noise floor.
- SGD still diverges (the 5 shown SGD rows have RMSE 5.9e10 to 3.5e11), and LinearSVR failed to converge in all 55 fits (11 imputers x 5 fits). Imputing does nothing for the scale problem from A.
- KNNImputer ran on unscaled features, so its neighbor distances were dominated by Healthy life expectancy (standard deviation 7.5, against 0.08 to 0.19 for the features on a 0 to 1 scale). Its "nearest neighbors" were mostly just countries with similar life expectancy.
- **Carry forward**
  - Imputing lets the combined experiments keep all 1,559 rows without a visible cost in RMSE.
  - Step order matters for KNNImputer: scale before it (sklearn's scalers pass NaN through), or its distances stay life expectancy distances.
  - Compare imputed pipelines with each other, not with A, B, D, or E, since they are scored on different rows.

# Experiment D | Transform Only

In [19]:
experimentD_model_output_dir = MODEL_OUTPUT_DIR / 'D-TRANSFORM-DropRowsWithMissingValues'
experimentD_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> transforms don't fill missing values, so rows with missing values are dropped as in Experiments A and B. This also means D is scored on the same rows as A, so the two compare directly

In [20]:
X_experimentD_clean, y_experimentD_clean = drop_missing_rows(X, y)

## PreProcessing

| impute      | transform                                                                                  | scale       |
|-------------|--------------------------------------------------------------------------------------------|-------------|
| passthrough | yeo-johnson, box-cox, log, sqrt, power 0.25, 0.75, 2, 3 (each on all 6 skewed columns)     | passthrough |

> Every variant applies one transform to **all 6 skewed columns** (|skew| ≥ 0.5, the Q1B rule): Social support, Healthy life expectancy, Freedom, Generosity, Perceptions of corruption, and Negative affect. Log GDP per capita (-0.29) and Positive affect (-0.35) are roughly symmetric and pass through unchanged. Transforming only some of the skewed columns is left to a later combinations experiment
> - **PowerTransformer** learns its own power (λ) per column on each training fold, so it bends left- and right-skewed columns in opposite directions. `standardize=False` keeps scaling out of this experiment
> - **Fixed transforms** use the same formula on every column, so each one works against one skew direction: log, sqrt, and powers below 1 fix right skew (Generosity, Negative affect) but worsen left skew (the other 4); powers above 1 do the reverse
> - **MinShift:** Generosity is ≤ 0 in 780 of the 1,374 rows. Log, sqrt, and fractional powers would return NaN, squaring would map -0.3 and 0.3 to the same value, and Box-Cox needs x > 0. MinShift learns the training fold's minimum and shifts columns with values ≤ 0 up first (floor 1 before log and Box-Cox, floor 0 before the powers). A validation value below the training fold's minimum is clipped to the floor. The other 5 skewed columns are already positive and are not shifted

In [21]:
# the skewed columns, using the same rule as Q1B: |skew| >= 0.5 on the training set
SKEW_THRESHOLD = 0.5
SKEWED = [c for c in X.columns if abs(X[c].skew()) >= SKEW_THRESHOLD]

experimentD_preProcessing = [
	# PowerTransformer: learns a power per column (Box-Cox needs x > 0, so MinShift first for Generosity)
	create_preprocessing(preprocessing={ "transform": on_columns(PowerTransformer(method="yeo-johnson", standardize=False), SKEWED) }),
	create_preprocessing(preprocessing={ "transform": on_columns(make_pipeline(MinShift(floor=1), PowerTransformer(method="box-cox", standardize=False)), SKEWED) }),

	# log, sqrt, and powers below 1 (MinShift first, since Generosity has values <= 0)
	create_preprocessing(preprocessing={ "transform": on_columns(make_pipeline(MinShift(floor=1), FunctionTransformer(np.log)), SKEWED) }),
	create_preprocessing(preprocessing={ "transform": on_columns(make_pipeline(MinShift(floor=0), FunctionTransformer(np.sqrt)), SKEWED) }),
	create_preprocessing(preprocessing={ "transform": on_columns(make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 0.25})), SKEWED) }),
	create_preprocessing(preprocessing={ "transform": on_columns(make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 0.75})), SKEWED) }),

	# powers above 1 (MinShift first, since squaring would map -0.3 and 0.3 to the same value)
	create_preprocessing(preprocessing={ "transform": on_columns(make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 2})), SKEWED) }),
	create_preprocessing(preprocessing={ "transform": on_columns(make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 3})), SKEWED) }),
]

## Build Pipelines

In [22]:
experimentD_pipelines = build_pipelines(preprocessings=experimentD_preProcessing, models=MODELS, seed=SEED)
# display_pipeline_grid(experimentD_pipelines)

## Cross-Validation Model Fitting

> cross_validate is a function that fits immediately; GridSearchCV is an object that waits for .fit()

In [23]:
experimentD_df = run_experiment(
    experimentD_pipelines,
    X=X_experimentD_clean,
    y=y_experimentD_clean,
    cv=CV,
    scorers=SCORERS,
    output_dir=experimentD_model_output_dir,
    experiment="D",
    cleaning="drop_missing_rows",
)
experimentD_df

/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the numb

,experiment,model,cleaning,impute,transform,scale,rmse,mse,mae,path
0,D,ridge,drop_missing_rows,none,ColumnTransformer(PowerTransformer(method=yeo-...,none,0.5260,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
1,D,linear_regression,drop_missing_rows,none,ColumnTransformer(PowerTransformer(method=yeo-...,none,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
2,D,linear_regression,drop_missing_rows,none,ColumnTransformer(MinShift(floor=1)+PowerTrans...,none,0.5270,0.2780,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
3,D,ridge,drop_missing_rows,none,ColumnTransformer(MinShift(floor=1)+PowerTrans...,none,0.5310,0.2820,0.4140,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
4,D,ridge,drop_missing_rows,none,ColumnTransformer(MinShift(floor=0)+FunctionTr...,none,0.5330,0.2850,0.4130,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
5,D,linear_regression,drop_missing_rows,none,ColumnTransformer(MinShift(floor=0)+FunctionTr...,none,0.5340,0.2850,0.4130,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
6,D,ridge,drop_missing_rows,none,ColumnTransformer(MinShift(floor=0)+FunctionTr...,none,0.5390,0.2910,0.4170,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
7,D,linear_regression,drop_missing_rows,none,ColumnTransformer(MinShift(floor=0)+FunctionTr...,none,0.5400,0.2910,0.4170,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
8,D,ridge,drop_missing_rows,none,ColumnTransformer(MinShift(floor=0)+FunctionTr...,none,0.5490,0.3010,0.4260,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
9,D,linear_regression,drop_missing_rows,none,ColumnTransformer(MinShift(floor=0)+FunctionTr...,none,0.5490,0.3010,0.4240,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Experiment D Results

- Yeo-Johnson on the 6 skewed columns gives the best scores so far: ridge 0.526, linear_regression 0.527 (MAE 0.409, against 0.422 in A). That is about 0.02 better than A, roughly 10 times the noise floor. Box-Cox (after MinShift) ties for linear_regression (0.527) but drops to 0.531 with ridge.
- The two powers above 1 also beat A: linear_regression 0.534 (cube) and 0.540 (square), ridge 0.533 and 0.539.
- log (0.560), sqrt, and the powers below 1 (0.549 to 0.554) are worse than A. Only Generosity (skew 0.83) and Negative affect (0.70) are right-skewed; Social support (-1.14), Healthy life expectancy (-0.72), Freedom (-0.61), and Perceptions of corruption (-1.54) are left-skewed, so a compressing transform makes 4 of the 6 columns more skewed. Powers above 1 do the reverse (help 4, hurt 2), and the fitted power transforms pick a direction per column, which is why they win.
- Without scaling, the other models break: svr_linear hit its 1,000,000 max_iter cap on 4 of the 8 transforms (20 "Solver terminated early" warnings = 4 transforms x 5 fits) and scored 13.7 to 608,060 on those; LinearSVR failed to converge in all 40 fits (0.555 to 1.418); SGD diverged on 5 of 8 (RMSE 6.3e7 to 1.6e20) and was 0.589 at best.
- Lasso and ElasticNet (0.609 to 1.105) are still over-penalized at alpha=1.
- **Carry forward**
  - Use Yeo-Johnson on the skewed columns as the default transform. Drop log, sqrt, p0.25, and p0.75 as whole-set transforms.
  - Transforms can make values huge (cubed Healthy life expectancy reaches about 458,000, 77.1 cubed), so scale after transforming: impute, then transform, then scale, then the model.
  - Box-Cox's ridge penalty (0.531 vs 0.527) should go away once a scaler follows the transform; check it in the combined runs instead of assuming.

# Experiment E | Transform Combinations Only

> column specific, skew aware, combinations matrix.

EX:
- Right Skew
	- columns = [1,2]
	- transforms = [a,b,c]
- Left Skew
	- columns = [3,4]
	- transforms = [d,e]

In [24]:
from itertools import product, combinations
import pandas as pd

groups = {
    "right_skew": {"columns": [1, 2], "transforms": ["a", "b", "c"]},
    "left_skew":  {"columns": [3, 4], "transforms": ["d", "e"]},
}

In [25]:
# V1 - less comprehensive (fallback option)
# Map every column to the transforms allowed for its skew group
col_options = {col: g["transforms"]
               for g in groups.values()
               for col in g["columns"]}

matrix = pd.DataFrame(
    list(product(*col_options.values())),
    columns=list(col_options),
)
print(matrix)
print(len(matrix))

    1  2  3  4
0   a  a  d  d
1   a  a  d  e
2   a  a  e  d
3   a  a  e  e
4   a  b  d  d
5   a  b  d  e
6   a  b  e  d
7   a  b  e  e
8   a  c  d  d
9   a  c  d  e
10  a  c  e  d
11  a  c  e  e
12  b  a  d  d
13  b  a  d  e
14  b  a  e  d
15  b  a  e  e
16  b  b  d  d
17  b  b  d  e
18  b  b  e  d
19  b  b  e  e
20  b  c  d  d
21  b  c  d  e
22  b  c  e  d
23  b  c  e  e
24  c  a  d  d
25  c  a  d  e
26  c  a  e  d
27  c  a  e  e
28  c  b  d  d
29  c  b  d  e
30  c  b  e  d
31  c  b  e  e
32  c  c  d  d
33  c  c  d  e
34  c  c  e  d
35  c  c  e  e
36


In [26]:
# V2 - fully exhaustive (WHAT I WANT TO DO)
col_options = {col: g["transforms"]
               for g in groups.values()
               for col in g["columns"]}
cols = list(col_options)

rows = []
for k in range(1, len(cols) + 1):                 # how many columns change
    for subset in combinations(cols, k):          # which columns change
        for choice in product(*(col_options[c] for c in subset)):
            row = {c: "none" for c in cols}       # everything else untouched
            row.update(zip(subset, choice))
            row["n_changed"] = k
            rows.append(row)

matrix = pd.DataFrame(rows)
print(matrix.groupby("n_changed").size())
print(matrix)
print(len(matrix))

n_changed
1    10
2    37
3    60
4    36
dtype: int64
        1     2     3     4  n_changed
0       a  none  none  none          1
1       b  none  none  none          1
2       c  none  none  none          1
3    none     a  none  none          1
4    none     b  none  none          1
..    ...   ...   ...   ...        ...
138     c     b     e     e          4
139     c     c     d     d          4
140     c     c     d     e          4
141     c     c     e     d          4
142     c     c     e     e          4

[143 rows x 5 columns]
143


In [27]:
experimentE_model_output_dir = MODEL_OUTPUT_DIR / 'E-TRANSFORM-COMBINATIONS-DropRowsWithMissingValues'
experimentE_model_output_dir.mkdir(parents=True, exist_ok=True)

# every result is appended here as soon as it is scored, so an interrupted run resumes where it stopped.
# the settings behind it are saved next to it (results.settings.json); if they change, it is deleted automatically
experimentE_csv_path = experimentE_model_output_dir / 'results.csv'

## Cleaning

> transforms don't fill missing values, so rows with missing values are dropped as in Experiments A, B, and D. E is scored on the same rows and folds as A and D, so its rows where every skewed column gets Yeo-Johnson (or Box-Cox) should reproduce D's scores exactly

In [28]:
X_experimentE_clean, y_experimentE_clean = drop_missing_rows(X, y)

## PreProcessing

| impute      | transform                                                                                                       | scale       |
|-------------|-----------------------------------------------------------------------------------------------------------------|-------------|
| passthrough | every combination of: none or a skew-aware transform, chosen separately for each of the 6 skewed columns       | passthrough |

> The V2 matrix above, with the real columns and the transforms Experiment D tested:
> - **Right skew** (Generosity, Negative affect): none, log, sqrt, power 0.25, power 0.75, Yeo-Johnson, Box-Cox (7 options)
> - **Left skew** (Social support, Healthy life expectancy, Freedom, Perceptions of corruption): none, power 2, power 3, Yeo-Johnson, Box-Cox (5 options)
> - Yeo-Johnson and Box-Cox are in both groups, since they learn their own power per column. The fixed transforms only appear in the group whose skew they fix (D showed the wrong direction hurts)
> - Every column picking "none" is Experiment A, so it is left out: 7² × 5⁴ - 1 = **30,624 combinations**
> - Log GDP per capita and Positive affect are not skewed and pass through unchanged
> - **Models:** only linear_regression and ridge. On unscaled data the other five don't give meaningful scores (SGD diverges, LinearSVR and svr_linear don't converge, Lasso and ElasticNet are held down by alpha=1), so 30,624 × 2 = **61,248 pipelines**
> - The `transform` column holds a short code (ex: `Social=p2,Life exp=yj,...`), and the results get one extra column per skewed feature with its transform, plus `n_changed`

In [29]:
# the same transformers as Experiment D, keyed by the short code used in the results
TRANSFORMS = {
	"yj":    PowerTransformer(method="yeo-johnson", standardize=False),
	"bc":    make_pipeline(MinShift(floor=1), PowerTransformer(method="box-cox", standardize=False)),
	"log":   make_pipeline(MinShift(floor=1), FunctionTransformer(np.log)),
	"sqrt":  make_pipeline(MinShift(floor=0), FunctionTransformer(np.sqrt)),
	"p0.25": make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 0.25})),
	"p0.75": make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 0.75})),
	"p2":    make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 2})),
	"p3":    make_pipeline(MinShift(floor=0), FunctionTransformer(raise_to_power, kw_args={"exponent": 3})),
}

# SKEWED (|skew| >= 0.5) comes from Experiment D; the sign of the skew picks the group
experimentE_groups = {
	"right_skew": {"columns": [c for c in SKEWED if X[c].skew() > 0], "transforms": ["log", "sqrt", "p0.25", "p0.75", "yj", "bc"]},
	"left_skew":  {"columns": [c for c in SKEWED if X[c].skew() < 0], "transforms": ["p2", "p3", "yj", "bc"]},
}

experimentE_models = {name: MODELS[name] for name in ["linear_regression", "ridge"]}

experimentE_combinations = skew_aware_combinations(experimentE_groups)
for group, settings in experimentE_groups.items():
	print(f"{group}: {settings['columns']}")
print(experimentE_combinations.groupby("n_changed").size().to_string())
print(f"{len(experimentE_combinations):,} combinations × {len(experimentE_models)} models = {len(experimentE_combinations) * len(experimentE_models):,} pipelines")

right_skew: ['Generosity', 'Negative affect']
left_skew: ['Social support', 'Healthy life expectancy at birth', 'Freedom to make life choices', 'Perceptions of corruption']
n_changed
1       28
2      324
3     1984
4     6784
5    12288
6     9216
30,624 combinations × 2 models = 61,248 pipelines


## Build Pipelines

> the 61,248 pipelines are built one at a time inside the sweep instead of all up front, and rendering every one would freeze the notebook, so this shows a sample: the first combination, a mixed one, and every column on Yeo-Johnson (which should match Experiment D)

In [30]:
experimentE_features = [c for c in experimentE_combinations.columns if c != "n_changed"]
experimentE_sample = [
	experimentE_combinations.iloc[0],
	experimentE_combinations[experimentE_combinations["n_changed"] == 3].iloc[0],
	experimentE_combinations[(experimentE_combinations[experimentE_features] == "yj").all(axis=1)].iloc[0],
]
experimentE_sample_pipelines = {
	(model_name, combination_code(row[experimentE_features].to_dict())):
		combination_pipeline(row[experimentE_features].to_dict(), transforms=TRANSFORMS, model=model, seed=SEED)
	for row in experimentE_sample
	for model_name, model in experimentE_models.items()
}
# display_pipeline_grid(experimentE_sample_pipelines, columns=2)

## Cross-Validation Model Fitting

> cross_validate is a function that fits immediately; GridSearchCV is an object that waits for .fit()
>
> - every result is appended to `results.csv` as soon as it is scored. If the kernel dies or the run is interrupted, rerun this cell and it skips everything already in the csv
> - the settings behind the csv (the matrix, each transform's settings, the models, seed, folds, metrics, and the data) are saved in `results.settings.json`. If any of them change, the csv is deleted automatically and the sweep starts over, and the cell prints which settings changed
> - progress prints every 500 pipelines with an estimate of the time left
> - no models are saved during the sweep; at the end the best 10 per model are refit on all the rows and saved (their `path` is filled in, every other row's is empty)
> - to stop macOS from sleeping during the run, start Jupyter with `caffeinate -i jupyter lab` (or run `caffeinate -i` in a terminal while it works)

In [31]:
experimentE_df = run_combinations(
	experimentE_combinations,
	transforms=TRANSFORMS,
	models=experimentE_models,
	X=X_experimentE_clean,
	y=y_experimentE_clean,
	cv=CV,
	scorers=SCORERS,
	seed=SEED,
	csv_path=experimentE_csv_path,
	output_dir=experimentE_model_output_dir,
	experiment="E",
	cleaning="drop_missing_rows",
	top_n=10,
)
experimentE_df

0 of 61,248 pipelines already in results.csv
500 / 61,248 done, 0.1 min elapsed, ~16.7 min left
1,000 / 61,248 done, 0.3 min elapsed, ~17.5 min left
1,500 / 61,248 done, 0.4 min elapsed, ~17.8 min left
2,000 / 61,248 done, 0.6 min elapsed, ~19.1 min left
2,500 / 61,248 done, 0.9 min elapsed, ~20.1 min left
3,000 / 61,248 done, 1.0 min elapsed, ~19.9 min left
3,500 / 61,248 done, 1.2 min elapsed, ~20.1 min left
4,000 / 61,248 done, 1.4 min elapsed, ~19.9 min left
4,500 / 61,248 done, 1.6 min elapsed, ~19.6 min left
5,000 / 61,248 done, 1.7 min elapsed, ~19.6 min left
5,500 / 61,248 done, 1.9 min elapsed, ~19.2 min left
6,000 / 61,248 done, 2.1 min elapsed, ~19.1 min left
6,500 / 61,248 done, 2.2 min elapsed, ~18.8 min left
7,000 / 61,248 done, 2.4 min elapsed, ~18.6 min left
7,500 / 61,248 done, 2.6 min elapsed, ~18.5 min left
8,000 / 61,248 done, 2.8 min elapsed, ~18.3 min left
8,500 / 61,248 done, 2.9 min elapsed, ~18.3 min left
9,000 / 61,248 done, 3.1 min elapsed, ~18.1 min left
9,5

,experiment,model,cleaning,impute,transform,scale,rmse,mse,mae,path,Generosity,Neg affect,Social,Life exp,Freedom,Corruption,n_changed
0,E,ridge,drop_missing_rows,none,"Generosity=yj,Neg affect=yj,Social=yj,Life exp...",none,0.5260,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...,yj,yj,yj,bc,bc,p3,6
1,E,ridge,drop_missing_rows,none,"Generosity=yj,Neg affect=yj,Social=yj,Life exp...",none,0.5260,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...,yj,yj,yj,yj,bc,p3,6
2,E,ridge,drop_missing_rows,none,"Generosity=bc,Neg affect=yj,Social=yj,Life exp...",none,0.5260,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...,bc,yj,yj,bc,bc,p3,6
3,E,ridge,drop_missing_rows,none,"Generosity=bc,Neg affect=yj,Social=yj,Life exp...",none,0.5260,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...,bc,yj,yj,yj,bc,p3,6
4,E,ridge,drop_missing_rows,none,"Generosity=yj,Neg affect=yj,Social=yj,Life exp...",none,0.5260,0.2770,0.4090,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...,yj,yj,yj,bc,bc,yj,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61243,E,linear_regression,drop_missing_rows,none,"Generosity=none,Neg affect=yj,Social=none,Life...",none,0.5460,0.2990,0.4220,,none,yj,none,none,none,none,1
61244,E,linear_regression,drop_missing_rows,none,"Generosity=none,Neg affect=bc,Social=none,Life...",none,0.5460,0.2990,0.4220,,none,bc,none,none,none,none,1
61245,E,linear_regression,drop_missing_rows,none,"Generosity=none,Neg affect=sqrt,Social=none,Li...",none,0.5460,0.2990,0.4220,,none,sqrt,none,none,none,none,1
61246,E,ridge,drop_missing_rows,none,"Generosity=none,Neg affect=bc,Social=none,Life...",none,0.5470,0.2990,0.4230,,none,bc,none,none,none,none,1


## Experiment E Results

- 30,624 combinations x 2 models = 61,248 pipelines. Each skewed column gets "none" or one of its group's transforms (right-skewed: log, sqrt, p0.25, p0.75, yj, bc; left-skewed: p2, p3, yj, bc).
- Best combinations: ridge 0.5264 (Generosity=yj, Neg affect=yj, Social=yj, Life exp=bc, Freedom=bc, Corruption=p3) and linear_regression 0.5266 (yj, log, yj, bc, p2, p3). All-Yeo-Johnson scores 0.5265 (ridge) / 0.5267 (linear_regression), matching D, so the best per-column choice beats it by 0.0001: a tie. 37 ridge and 97 linear_regression combinations beat all-Yeo-Johnson, none by 0.002 or more, and 2,575 / 5,435 combinations sit within 0.002 of the best.
- Per column (mean RMSE across every combination that uses each transform, linear_regression): Social (yj 0.5298 vs none 0.5384, a 0.0086 gap) and Life exp (bc / yj 0.5309 vs none 0.5371, 0.0062) carry most of the gain. Corruption (0.0023) and Generosity (0.0020) are right at the noise floor. Neg affect (0.0002) and Freedom (0.0002) don't matter: Freedom's min RMSE is 0.5266 whether it is transformed or not. Ridge shows the same ranking.
- Transforming only Social and Life exp (yj) gives 0.5309, which is 0.004 short of all-Yeo-Johnson; adding Corruption and Generosity (yj on 4 columns) gives 0.5269, a tie with all 6.
- Ridge with Box-Cox on Social averages 0.5341 against 0.5299 for yj (linear_regression: 0.5301 vs 0.5298), the same ridge and Box-Cox penalty seen in D.
- **Carry forward**
  - The per-column search buys nothing over all-Yeo-Johnson (0.0001), so the combined experiments should use one Yeo-Johnson step on the skewed columns instead of this 30,624-combination grid. That keeps room in the grid for 7 models x imputers x scalers.
  - If the transform has to be trimmed, keep Social, Life exp, Corruption, and Generosity; Freedom and Neg affect can pass through.
  - E only tested linear_regression and ridge without scaling, so the ranking still needs to be confirmed with a scaler and the other 5 models.